# Icon: every expression a grammar allows

A calculator's grammar is four rules: an expression is a term, a term a factor, a factor an element, an element `x`, `y`, `z`, a number or a parenthesised expression. In Icon each rule is one procedure, and a procedure can be a *generator*: it `suspend`s each alternative in turn and Icon backtracks into it for the next. The first cell walks the grammar exhaustively, every expression of at most N tokens; the reversible assignment `<-` counts tokens on the path and undoes the count when the path is backtracked. The second cell walks it at random instead, one expression a call. The last cell runs both.

In [ ]:
global tokens, limit

procedure e_token(s)
    suspend (tokens <- tokens + 1, tokens <= limit, s);
end

procedure e_element()
    suspend ( e_token("x")
            | e_token("y")
            | e_token("z")
            | e_token("1")
            | (e_token("(") || e_expression() || e_token(")"))
            );
end

procedure e_factor()
    suspend ( e_element()
            | (e_token("+") || e_factor())
            | (e_token("-") || e_factor())
            );
end

procedure e_term()
    suspend ( e_factor()
            | (e_factor() || e_token("*") || e_term())
            | (e_factor() || e_token("/") || e_term())
            );
end

procedure e_expression()
    suspend ( e_term()
            | (e_term() || e_token("+") || e_expression())
            | (e_term() || e_token("-") || e_expression())
            );
end

In [ ]:
procedure r_element()
    local r;
    r := ?100;
    return ( (r <= 20, "x")
           | (r <= 40, "y")
           | (r <= 60, "z")
           | (r <= 80, string(?100))
           | ("(" || r_expression() || ")")
           );
end

procedure r_factor()
    local r;
    r := ?100;
    return ( (r <= 80, r_element())
           | (r <= 90, "+" || r_factor())
           | ("-" || r_factor())
           );
end

procedure r_term()
    local r;
    r := ?100;
    return ( (r <= 70, r_factor())
           | (r <= 85, r_factor() || "*" || r_term())
           | (r_factor() || "/" || r_term())
           );
end

procedure r_expression()
    local r;
    r := ?100;
    return ( (r <= 70, r_term())
           | (r <= 85, r_term() || "+" || r_expression())
           | (r_term() || "-" || r_expression())
           );
end

In [ ]:
procedure main()
   local n;
   limit := 3;
   tokens := 0;
   n := 0;
   every write(e_expression()) do n +:= 1;
   write(n, " expressions of at most ", limit, " tokens");
   &random := 1;
   every 1 to 8 do write(r_expression());
end